In [ ]:
import os

files = os.listdir("FingeringFiles")
print(f"sum files: {len(files)}")

In [ ]:
import pandas as pd

columns = [
    "note_id", "onset_time", "offset_time", "spelled_pitch",
    "onset_velocity", "offset_velocity", "channel", "finger_number"
]

In [ ]:
import os
import glob
import math
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import music21

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

COLUMNS = [
    "note_id",
    "onset_time",
    "offset_time",
    "spelled_pitch",
    "onset_velocity",
    "offset_velocity",
    "channel",
    "finger_number",
]

print("Imports hazır.")

In [ ]:
SEED = 59

DATA_DIR = "FingeringFiles"
LIST_PATH = "List.csv"

SEQUENCE_LENGTH = 64
STRIDE = 64

BATCH_SIZE = 32

LEARNING_RATE = 1e-3
NUM_EPOCHS = 20

HIDDEN_DIM = 128
NHEAD = 4
NUM_LAYERS = 3
DROPOUT = 0.1

CHECKPOINT_PATH = "best_transformer_baseline.pt"

In [ ]:
import os
import random
import numpy as np
import torch

def set_seed(SEED):
    random.seed(SEED)
    os.environ['PYTHONHASHSEED'] = str(SEED)

    np.random.seed(SEED)
    
    torch.manual_seed(SEED)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Fonksiyonu çağırarak tüm sistemi kilitliyoruz
set_seed(59)
print("Tüm rastgelelik motorları (Python, NumPy, PyTorch, CUDA) 59 seed değerine kilitlendi.")

In [ ]:
sample_file = "FingeringFiles/001-1_fingering.txt"

df_sample = pd.read_csv(
    sample_file, 
    sep=r"\s+", #bu regex o verisetindeki boşlukve tabları  ayarlıyor
    comment="/", 
    header=None, 
    names=columns
)

df_sample.head()

In [ ]:
df_list = pd.read_csv("List.csv")

composer_counts = df_list["Composer"].value_counts()

print(f"eser sayısı: {len(df_list)}")
print(composer_counts)

In [ ]:
plt.figure(figsize=(10, 4))
composer_counts.plot(kind="bar", color="royalblue", edgecolor="black")
plt.title("sanatçı dağılımı")
plt.xlabel("piyanist")
plt.ylabel("eser")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df_sample["finger_clean"] = df_sample["finger_number"].astype(str).apply(lambda x: x.split("_")[0])
df_sample["finger"] = pd.to_numeric(
    df_sample["finger_clean"],
    errors="coerce"
)
df_sample["midi_pitch"] = df_sample["spelled_pitch"].apply(lambda p: music21.pitch.Pitch(p).midi)
df_sample[["onset_time", "spelled_pitch", "midi_pitch", "channel", "finger"]].head()

In [ ]:
all_files = sorted(
    glob.glob(
        os.path.join(DATA_DIR, "*_fingering.txt")
    )
)

print(f"Fingering dosyası sayısı: {len(all_files)}")
 
records = []
for file_path in all_files:
    df_temp = pd.read_csv(file_path, sep=r"\s+", comment="/", header=None, usecols=[6, 7])
    records.append(df_temp)

all_fingers_df = pd.concat(records, ignore_index=True)
all_fingers_df.columns = ["channel", "finger_number"]
total_notes = len(all_fingers_df)
substitution_mask = all_fingers_df["finger_number"].astype(str).str.contains("_")
substitution_count = substitution_mask.sum()
normal_count = total_notes - substitution_count
substitution_ratio = (substitution_count / total_notes) * 100
print(f"okunan dosya sayısı: {len(records)} / {len(all_files)}")
print(f"okunan nota sayısı: {total_notes:,}")
print(f"Normal Fingering: {normal_count:,}")
print(f"Finger Substitution İçeren Nota: {substitution_count:,}")
print(f"Substitution Oranı: %{substitution_ratio:.2f}")
all_fingers_df.head()

In [ ]:
all_fingers_df["finger_clean"] = (
    all_fingers_df["finger_number"]
    .astype(str)
    .apply(lambda x: x.split("_")[0])
)

all_fingers_df["finger"] = pd.to_numeric(all_fingers_df["finger_clean"], errors="coerce")
all_fingers_df = all_fingers_df.dropna(subset=["finger"]).copy()
all_fingers_df["finger"] = all_fingers_df["finger"].astype(int)

right_hand_counts = (
    all_fingers_df[all_fingers_df["channel"] == 0]["finger"]
    .value_counts()
    .sort_index()
)

left_hand_counts = (
    all_fingers_df[all_fingers_df["channel"] == 1]["finger"]
 .abs()   
    .value_counts()
    .sort_index()
)

print("sağ el parmak sayıları")
print(right_hand_counts)

print("\nsol el parmak sayıları")
print(left_hand_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(right_hand_counts.index, right_hand_counts.values, color="royalblue", edgecolor="black")
axes[0].set_title("sağ el")
axes[0].set_xlabel("parmak no")
axes[0].set_ylabel("nota sayısı")
axes[0].set_xticks(range(1, 6))
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

axes[1].bar(left_hand_counts.index, left_hand_counts.values, color="crimson", edgecolor="black")
axes[1].set_title("sol el")
axes[1].set_xlabel("parmak no")
axes[1].set_ylabel("nota sayısı")
axes[1].set_xticks(range(1, 6))
axes[1].grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
piece_ids = sorted(list(set([os.path.basename(f).split("-")[0] for f in all_files])))
print(f"Toplam farklı eser sayısı: {len(piece_ids)}")

random.seed(SEED)
shuffled_pieces = piece_ids.copy()
random.shuffle(shuffled_pieces)
#validation eklendi
train_idx = int(len(shuffled_pieces) * 0.70)
val_idx = int(len(shuffled_pieces) * 0.85)

train_pieces = set(shuffled_pieces[:train_idx])
val_pieces = set(shuffled_pieces[train_idx:val_idx])
test_pieces = set(shuffled_pieces[val_idx:])

train_files = [f for f in all_files if os.path.basename(f).split("-")[0] in train_pieces]
val_files = [f for f in all_files if os.path.basename(f).split("-")[0] in val_pieces]
test_files = [f for f in all_files if os.path.basename(f).split("-")[0] in test_pieces]

print(f"Traindeki eserler: {len(train_pieces)} | İlgili Dosya Sayısı: {len(train_files)}")
print(f"Validation'daki eserler: {len(val_pieces)} | İlgili Dosya Sayısı: {len(val_files)}")
print(f"Testteki eserler: {len(test_pieces)} | İlgili Dosya Sayısı: {len(test_files)}")

In [ ]:
# ============================================================
# ADIM 11 — SPLIT INTEGRITY / LEAKAGE CHECK
# ============================================================

# 1. Hiçbir eser iki farklı split'te bulunmamalı
assert train_pieces.isdisjoint(val_pieces), \
    "HATA: Train ve Validation arasında ortak eser var!"

assert train_pieces.isdisjoint(test_pieces), \
    "HATA: Train ve Test arasında ortak eser var!"

assert val_pieces.isdisjoint(test_pieces), \
    "HATA: Validation ve Test arasında ortak eser var!"


# 2. Bütün eserler tam olarak bir split'e dağılmış olmalı
assert (
    len(train_pieces)
    + len(val_pieces)
    + len(test_pieces)
    == len(piece_ids)
), "HATA: Piece sayıları toplamı dataset ile uyuşmuyor!"


# 3. Bütün fingering dosyaları tam olarak bir split'e dağılmış olmalı
assert (
    len(train_files)
    + len(val_files)
    + len(test_files)
    == len(all_files)
), "HATA: Fingering dosyaları splitlerde eksik veya tekrarlı!"


print("✅ SPLIT INTEGRITY CHECK BAŞARILI")
print()
print(f"Toplam eser       : {len(piece_ids)}")
print(f"Train eser        : {len(train_pieces)}")
print(f"Validation eser   : {len(val_pieces)}")
print(f"Test eser         : {len(test_pieces)}")
print()
print(f"Toplam dosya      : {len(all_files)}")
print(f"Train dosya       : {len(train_files)}")
print(f"Validation dosya  : {len(val_files)}")
print(f"Test dosya        : {len(test_files)}")

In [ ]:
import pandas as pd
import numpy as np
import music21
import torch
from torch.utils.data import Dataset

# 1. ORTAK VERİ TEMİZLEME FONKSİYONU
def load_and_clean_pig_file(file_path):

    df = pd.read_csv(
        file_path,
        sep=r"\s+",
        comment="/",
        header=None,
        names=columns
    )

    raw_rows = len(df)

    # ----------------------------
    # Sayısal dönüşümler
    # ----------------------------
    df["onset_time"] = pd.to_numeric(
        df["onset_time"],
        errors="coerce"
    )

    df["offset_time"] = pd.to_numeric(
        df["offset_time"],
        errors="coerce"
    )

    df["channel"] = pd.to_numeric(
        df["channel"],
        errors="coerce"
    )

    # ----------------------------
    # Finger temizleme
    # ----------------------------
    df["finger_clean"] = (
        df["finger_number"]
        .astype(str)
        .str.split("_")
        .str[0]
    )

    df["finger_signed"] = pd.to_numeric(
        df["finger_clean"],
        errors="coerce"
    )

    # ----------------------------
    # Zorunlu alanları temizle
    # ----------------------------
    df = df.dropna(
        subset=[
            "onset_time",
            "offset_time",
            "channel",
            "finger_signed",
            "spelled_pitch"
        ]
    ).copy()

    # ----------------------------
    # Channel kontrol
    # ----------------------------
    assert df["channel"].isin([0, 1]).all()

    # ----------------------------
    # Finger kontrol
    # ----------------------------
    assert (
        df["finger_signed"].abs().isin([1, 2, 3, 4, 5])
    ).all()

    # ----------------------------
    # Hand/Finger tutarlılığı
    # ----------------------------
    right_hand_valid = (
        (df["channel"] == 0) &
        (df["finger_signed"] > 0)
    )

    left_hand_valid = (
        (df["channel"] == 1) &
        (df["finger_signed"] < 0)
    )

    assert (
        right_hand_valid | left_hand_valid
    ).all(), f"Hand/finger mismatch: {file_path}"

    # ----------------------------
    # Unsigned finger target
    # ----------------------------
    df["finger"] = (
        df["finger_signed"]
        .abs()
        .astype(int)
    )

    # ----------------------------
    # MIDI pitch
    # ----------------------------
    df["midi_pitch"] = df["spelled_pitch"].apply(
        lambda p: music21.pitch.Pitch(p).midi
    )

    # ----------------------------
    # Timing
    # ----------------------------
    df["duration"] = (
        df["offset_time"] -
        df["onset_time"]
    )

    assert (
        df["duration"] >= 0
    ).all()

    assert np.isfinite(
        df["onset_time"]
    ).all()

    assert np.isfinite(
        df["offset_time"]
    ).all()

    assert np.isfinite(
        df["duration"]
    ).all()

    # ----------------------------
    # Deterministic sorting
    # ----------------------------
    df = df.sort_values(
        ["onset_time", "note_id"],
        kind="mergesort"
    ).reset_index(drop=True)

    # ----------------------------
    # Relative timing
    # ----------------------------
    df["onset_diff"] = (
        df["onset_time"]
        .diff()
        .fillna(0.0)
    )

    assert np.isfinite(
        df["onset_diff"]
    ).all()

    assert np.isfinite(
        df["midi_pitch"]
    ).all()

    cleaned_rows = len(df)

    return (
        df,
        raw_rows,
        cleaned_rows
    )
# 2. DATASET SINIFI
class PianoSequenceDataset(Dataset):
    def __init__(self, file_list, sequence_length=64, feature_stats=None, verbose=True):
        self.sequence_length = sequence_length
        self.inputs = []
        self.target_hands = []
        self.target_fingers = []
        
        total_raw_rows = 0
        total_cleaned_rows = 0
        
        for file_path in file_list:
            df, raw_rows, cleaned_rows = load_and_clean_pig_file(file_path)
            total_raw_rows += raw_rows
            total_cleaned_rows += cleaned_rows
            
            X_features = df[["midi_pitch", "duration", "onset_diff"]].values

            if feature_stats is not None:
                X_features = (X_features - feature_stats['mean']) / feature_stats['std']
                
            Y_hand = df["channel"].values
            Y_finger = df["finger"].values - 1 
            
            for i in range(0, len(df) - sequence_length + 1, sequence_length):
                self.inputs.append(X_features[i : i + sequence_length])
                self.target_hands.append(Y_hand[i : i + sequence_length])
                self.target_fingers.append(Y_finger[i : i + sequence_length])
                
        if verbose:
            print(f"--- VERİ TEMİZLİK RAPORU ---")
            print(f"Toplam Ham Satır: {total_raw_rows:,}")
            print(f"Temizlik Sonrası Net Satır: {total_cleaned_rows:,}\n")

        self.inputs = torch.tensor(np.array(self.inputs), dtype=torch.float32)
        self.target_hands = torch.tensor(np.array(self.target_hands), dtype=torch.long)
        self.target_fingers = torch.tensor(np.array(self.target_fingers), dtype=torch.long)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.target_hands[idx], self.target_fingers[idx]

In [ ]:
# 3. İSTATİSTİK HESAPLAMA FONKSİYONU
def get_train_stats(file_list):
    all_features = []
    print("Eğitim seti üzerinden istatistikler hesaplanıyor...")
    for file_path in file_list:
        df, _, _ = load_and_clean_pig_file(file_path)
        X = df[["midi_pitch", "duration", "onset_diff"]].values
        all_features.append(X)
        
    all_features = np.vstack(all_features)
    stats = {
        'mean': np.mean(all_features, axis=0),
        'std': np.std(all_features, axis=0) + 1e-8 
    }
    return stats

# 4. İSTATİSTİKLERİ HESAPLAT VE SETLERİ OLUŞTUR
train_stats = get_train_stats(train_files)
print(f"Hesaplanan Ortalama (Mean): {train_stats['mean']}")
print(f"Hesaplanan Standart Sapma (Std): {train_stats['std']}\n")

train_dataset = PianoSequenceDataset(train_files, sequence_length=64, feature_stats=train_stats)
val_dataset = PianoSequenceDataset(val_files, sequence_length=64, feature_stats=train_stats)
test_dataset = PianoSequenceDataset(test_files, sequence_length=64, feature_stats=train_stats)

print(f"Eğitim (Train) Paket Sayısı: {len(train_dataset)}")
print(f"Doğrulama (Validation) Paket Sayısı: {len(val_dataset)}")
print(f"Test Paket Sayısı: {len(test_dataset)}")

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Eğitim DataLoader hazır! Toplam Batch (Yığın) Sayısı: {len(train_loader)}")
print(f"Test DataLoader hazır! Toplam Batch (Yığın) Sayısı: {len(test_loader)}")

for batch_inputs, batch_hands, batch_fingers in train_loader:
    print(f"\n--- BATCH (YIĞIN) TESTİ ---")
    print(f"Girdi Batch Boyutu (Batch_size, Sequence_length, Features): {batch_inputs.shape}")
    print(f"El Hedefleri Batch Boyutu: {batch_hands.shape}")
    print(f"Parmak Hedefleri Batch Boyutu: {batch_fingers.shape}")
    break

In [ ]:
import torch
import torch.nn as nn
import math

# 1. Konum Kodlayıcı (Notaların sırasını makineye hissettiren modül)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) 

    def forward(self, x):
        # Notaların üzerine sıra numarasını matematiksel bir dalga olarak ekler
        x = x + self.pe[:, :x.size(1), :]
        return x

# 2. Ana Makine: Multi-Task Transformer
class PianoFingeringModel(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=128, nhead=4, num_layers=3, dropout=0.1):
        super().__init__()
        
        # A. Gömme Katmanı (Ham 3 özelliği 128 boyutlu zengin bir beyin hücresine çevirir)
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.pos_encoder = PositionalEncoding(hidden_dim)
        
        # B. Transformer Encoder (Notaların birbiriyle bağ kurduğu, akorları anladığı yer)
        encoder_layers = nn.TransformerEncoderLayer(d_model=hidden_dim, 
                                                    nhead=nhead, 
                                                    dim_feedforward=hidden_dim*4, 
                                                    dropout=dropout, 
                                                    batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        # C. Çift Çıktı Kafası (Multi-Task Heads)
        # El Tahmini: 2 İhtimal (Sağ El=0, Sol El=1)
        self.hand_classifier = nn.Linear(hidden_dim, 2)
        
        # Parmak Tahmini: 5 İhtimal (0, 1, 2, 3, 4)
        self.finger_classifier = nn.Linear(hidden_dim, 5)

    def forward(self, x):
        # x'in geliş boyutu: (Batch=32, Sequence=64, Features=3)
        
        # 1. 3 özelliği 128'e genişlet ve notaların konumunu (sırasını) ekle
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        
        # 2. Müziği analiz et (Attention mekanizması çalışır)
        features = self.transformer_encoder(x) # Boyut: (32, 64, 128)
        
        # 3. İki farklı kararı ver
        hand_logits = self.hand_classifier(features)     # Çıktı Boyutu: (32, 64, 2)
        finger_logits = self.finger_classifier(features) # Çıktı Boyutu: (32, 64, 5)
        
        return hand_logits, finger_logits

In [ ]:
# 1. Modeli sahaya çağırıyoruz
model = PianoFingeringModel()

# 2. Taşıyıcı banttan (DataLoader) çektiğimiz o ilk 32'lik paketi (batch_inputs) modele veriyoruz
test_hand_out, test_finger_out = model(batch_inputs)

print("--- MODEL TEST RAPORU ---")
print(f"Modele giren veri boyutu: {batch_inputs.shape} -> (32 paket, 64 nota, 3 özellik)")
print(f"Modelin El Tahmini Boyutu: {test_hand_out.shape} -> (32 paket, 64 nota, 2 el ihtimali)")
print(f"Modelin Parmak Tahmini Boyutu: {test_finger_out.shape} -> (32 paket, 64 nota, 5 parmak ihtimali)")
print("\nMotor kusursuz çalışıyor! Eğitime (Training Loop) geçmeye hazırız.")

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import random
import torch

# --- EKLENEN KISIM: DATALOADER SEED AYARLARI ---
# Her bir DataLoader işçisinin (worker) seed'ini sabitleyen fonksiyon
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# PyTorch için rastgele sayı üreticisi (Generator) oluşturup kilitliyoruz
g = torch.Generator()
g.manual_seed(59)

# 1. TAŞIYICI BANTLARI (DATALOADER) 3 SETE GÖRE GÜNCELLİYORUZ
train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True, 
    worker_init_fn=seed_worker, 
    generator=g
)
# Test ve Validation setlerinde shuffle=False olduğu için 
# worker_init_fn ve generator eklemeye gerek yoktur.
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False) # Test setine final sınavına kadar dokunmuyoruz!

# 2. MODEL, HATA (LOSS) VE OPTİMİZASYON TANIMLAMALARI
# Ekran kartı (GPU/MPS) varsa onu, yoksa işlemciyi (CPU) kullansın
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = PianoFingeringModel().to(device)

# Parmak (5 sınıf) ve El (2 sınıf) için iki ayrı hata ölçücü
criterion_finger = nn.CrossEntropyLoss()
criterion_hand = nn.CrossEntropyLoss()

# Transformer modellerinde çok başarılı olan AdamW optimizatörü
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

# 3. EĞİTİM DÖNGÜSÜ (TRAINING LOOP)
num_epochs = 10
print(f"🚀 Eğitim Başlıyor... Kullanılan Cihaz: {device}\n")

for epoch in range(num_epochs):
    # --- A) EĞİTİM AŞAMASI ---
    model.train()
    total_train_loss = 0
    
    for batch_inputs, batch_hands, batch_fingers in train_loader:
        # Verileri donanıma gönderiyoruz
        batch_inputs = batch_inputs.to(device)
        batch_hands = batch_hands.to(device)
        batch_fingers = batch_fingers.to(device)
        
        optimizer.zero_grad() # Önceki turdan kalan hafızayı temizle
        
        # İleri Besleme (Tahmin yap)
        hand_logits, finger_logits = model(batch_inputs)
        
        # Loss hesaplamak için matrisleri 2 Boyuta düzleştiriyoruz (Flattening)
        loss_hand = criterion_hand(hand_logits.reshape(-1, 2), batch_hands.reshape(-1))
        loss_finger = criterion_finger(finger_logits.reshape(-1, 5), batch_fingers.reshape(-1))
        
        # Çoklu Görev (Multi-Task) Hatası: İki hatayı birleştiriyoruz
        total_loss = loss_finger + loss_hand
        
        # Geri Yayılım (Hatadan ders çıkarıp beyin hücrelerini güncelle)
        total_loss.backward()
        optimizer.step()
        
        total_train_loss += total_loss.item()
        
    # --- B) DOĞRULAMA (VALIDATION) AŞAMASI ---
    # Bu aşamada model öğrenmez (ağırlık güncellemez), sadece kendini test eder.
    model.eval()
    total_val_loss = 0
    correct_fingers = 0
    total_fingers = 0
    
    with torch.no_grad(): # Gradyan hesaplamayı kapatıyoruz
        for val_inputs, val_hands, val_fingers in val_loader:
            val_inputs = val_inputs.to(device)
            val_hands = val_hands.to(device)
            val_fingers = val_fingers.to(device)
            
            hand_logits, finger_logits = model(val_inputs)
            
            loss_h = criterion_hand(hand_logits.reshape(-1, 2), val_hands.reshape(-1))
            loss_f = criterion_finger(finger_logits.reshape(-1, 5), val_fingers.reshape(-1))
            total_val_loss += (loss_h + loss_f).item()
            
            # Parmak tahmini için başarı oranını (Accuracy) hesaplıyoruz
            predictions = torch.argmax(finger_logits, dim=-1)
            correct_fingers += (predictions == val_fingers).sum().item()
            total_fingers += val_fingers.numel()
            
    # Tur (Epoch) sonuçlarını yazdır
    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = (correct_fingers / total_fingers) * 100
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Parmak Başarısı: %{val_accuracy:.2f}")

print("\n🎉 Eğitim Tamamlandı!")

In [ ]:
# ============================================================
# DATA QUALITY AUDIT
# ============================================================

audit_results = []

audit_columns = [
    "note_id",
    "onset_time",
    "offset_time",
    "spelled_pitch",
    "channel",
    "finger_number"
]

for file_path in all_files:

    df = pd.read_csv(
        file_path,
        sep=r"\s+",
        comment="/",
        header=None,
        names=columns
    )

    # Ham satır
    raw_rows = len(df)

    # Sayısal dönüşümler
    onset = pd.to_numeric(df["onset_time"], errors="coerce")
    offset = pd.to_numeric(df["offset_time"], errors="coerce")
    channel = pd.to_numeric(df["channel"], errors="coerce")

    finger_clean = (
        df["finger_number"]
        .astype(str)
        .str.split("_")
        .str[0]
    )

    finger_signed = pd.to_numeric(
        finger_clean,
        errors="coerce"
    )

    # Kontroller
    missing_onset = onset.isna().sum()
    missing_offset = offset.isna().sum()
    missing_channel = channel.isna().sum()
    missing_finger = finger_signed.isna().sum()

    invalid_channel = (
        channel.notna() &
        ~channel.isin([0, 1])
    ).sum()

    invalid_finger = (
        finger_signed.notna() &
        ~finger_signed.abs().isin([1, 2, 3, 4, 5])
    ).sum()

    negative_duration = (
        (offset - onset) < 0
    ).sum()

    invalid_pitch = 0

    for pitch_value in df["spelled_pitch"]:
        try:
            pitch = music21.pitch.Pitch(pitch_value).midi
            if not np.isfinite(pitch):
                invalid_pitch += 1
        except Exception:
            invalid_pitch += 1

    valid_hand_finger = (
        channel.notna() &
        finger_signed.notna()
    )

    hand_finger_mismatch = (
        valid_hand_finger &
        ~(
            (
                (channel == 0) &
                (finger_signed > 0)
            )
            |
            (
                (channel == 1) &
                (finger_signed < 0)
            )
        )
    ).sum()

    audit_results.append({
        "file": os.path.basename(file_path),
        "raw_rows": raw_rows,
        "missing_onset": missing_onset,
        "missing_offset": missing_offset,
        "missing_channel": missing_channel,
        "missing_finger": missing_finger,
        "invalid_channel": invalid_channel,
        "invalid_finger": invalid_finger,
        "negative_duration": negative_duration,
        "invalid_pitch": invalid_pitch,
        "hand_finger_mismatch": hand_finger_mismatch
    })

audit_df = pd.DataFrame(audit_results)

print("=== DATA QUALITY AUDIT ===")
display(audit_df.sum(numeric_only=True).to_frame("count"))